In [1]:
import sys, os, json, gc
from pathlib import Path
from collections import Counter

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel, LoraConfig, prepare_model_for_kbit_training, get_peft_model
from trl import SFTTrainer, SFTConfig
from datasets import Dataset

print("python:", sys.executable)
print("torch:", torch.__version__, "cuda:", torch.cuda.is_available())

PROJECT = Path(r"C:\Users\shlok\projects\ddp-llm\parser")
DATA = PROJECT / "data"
CHECKPOINTS = PROJECT / "checkpoints"

# 500M target
MODEL_500M = "Qwen/Qwen2.5-0.5B-Instruct"
OUTPUT_500M_V2 = CHECKPOINTS / "qwen0p5b-qlora-v2_lite_reference"

assert (DATA / "train_v3.jsonl").exists()
assert (DATA / "test.jsonl").exists()
print("paths ok")

python: C:\Users\shlok\projects\ddp-llm\.venv\Scripts\python.exe
torch: 2.11.0+cu128 cuda: True
paths ok


In [2]:
# Same prompt that shipped with 3B v2 (see app config.py).
# Using this for a clean model-size comparison — same task, same data, same prompt.
SYSTEM_PROMPT_V3 = """You are a parser that converts a user's natural language response into a subset of the shown options.

The user is shown 4 options labeled A, B, C, D and gives feedback about which are close to what they want. Your job is to output which options the user views favorably.

Output format (JSON only, nothing else):
- A JSON list of the favored labels, e.g. ["A", "B"]
- [] if the user explicitly rejects ALL options ("none of these", "all wrong")
- "*" if the utterance is off-topic OR expresses no usable preference ("I don't know", "they all look the same", "I love football")

Rules:
- Any positive signal about an option means it goes in the list.
- "X is better than Y" endorses only X, not Y.
- "X and Y are both good, X is better" endorses both X and Y.
- Negations like "not D" or "anything but B" mean the remaining options go in the list.
- Questions like "is it A?" are treated as tentative endorsement of A.
- CRITICAL: If the utterance is only a short positive reaction that does NOT reference any option (e.g. "works great", "perfect", "nice one", "go with it", "that works", "cool"), output "*". A positive word alone is not enough. To count as an endorsement, the user must reference the options - either by letter (A, B, C, D), position (left, right, first, second, middle, last), quantifier over options (all, both, either, these, the ones, all four, both of them), or description that clearly points at an option.

Output ONLY the JSON. No explanation, no prose."""

print(f"prompt: {len(SYSTEM_PROMPT_V3)} chars")

prompt: 1480 chars


In [3]:
def load_jsonl(path):
    return [json.loads(l) for l in Path(path).read_text(encoding="utf-8").splitlines() if l.strip()]

def build_model(base_model_id, adapter_path):
    bnb = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
    )
    tok = AutoTokenizer.from_pretrained(base_model_id)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    base = AutoModelForCausalLM.from_pretrained(
        base_model_id, quantization_config=bnb, device_map="cuda",
    )
    m = PeftModel.from_pretrained(base, str(adapter_path))
    m.eval()
    return m, tok

def parse(model, tok, utterance, options, system_prompt=SYSTEM_PROMPT_V3):
    options_str = "[" + ", ".join(f'"{o}"' for o in options) + "]"
    user_msg = f"Options: {options_str}\nUser: {utterance}"
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_msg},
    ]
    inputs = tok.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors="pt", return_dict=True,
    ).to("cuda")
    with torch.no_grad():
        out = model.generate(
            **inputs, max_new_tokens=15, do_sample=False,
            pad_token_id=tok.eos_token_id,
        )
    raw = tok.decode(
        out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True,
    ).strip()
    if raw.startswith("```"):
        raw = raw.strip("`").lstrip("json").strip()
    try:
        parsed = json.loads(raw)
    except json.JSONDecodeError:
        return None, raw
    return parsed, raw

def labels_equal(a, b):
    if a == "*" or b == "*":
        return a == b
    if isinstance(a, list) and isinstance(b, list):
        return set(a) == set(b)
    return False

def free_gpu():
    gc.collect()
    torch.cuda.empty_cache()
    print(f"VRAM: {torch.cuda.memory_allocated() / 1e9:.2f} GB", flush=True)

print("helpers defined")

helpers defined


In [4]:
free_gpu()

# Load the SAME train.jsonl that was used for 3B v2 (1598 examples now, including
# both the v2 and v3 additions since we appended to the same file).
# For a truly clean comparison against 3B v2 (1451 examples), we'd need to
# roll back train.jsonl. But since the failed v3 additions were confirmation
# words which we're keeping as a documented limitation anyway, the 1598-example
# set is a fair "current production data" comparison.
train_data_full = load_jsonl(DATA / "train_v3.jsonl")
print(f"training on {len(train_data_full)} examples")

def to_chat(ex):
    options_str = "[" + ", ".join(f'"{o}"' for o in ex["options"]) + "]"
    user_msg = f"Options: {options_str}\nUser: {ex['utterance']}"
    assistant_msg = json.dumps(ex["label"])
    return {"messages": [
        {"role": "system", "content": SYSTEM_PROMPT_V3},
        {"role": "user", "content": user_msg},
        {"role": "assistant", "content": assistant_msg},
    ]}

train_ds = Dataset.from_list([to_chat(ex) for ex in train_data_full])
print(f"dataset size: {len(train_ds)}")

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)
tok = AutoTokenizer.from_pretrained(MODEL_500M)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

base = AutoModelForCausalLM.from_pretrained(
    MODEL_500M, quantization_config=bnb, device_map="cuda",
)
base = prepare_model_for_kbit_training(base)

# Same LoRA config as 1.5B/3B for apples-to-apples
lora_cfg = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
)
model = get_peft_model(base, lora_cfg)
model.print_trainable_parameters()

# 3 epochs (same as 1.5B/3B v2). 500M is small so we could go more,
# but keeping it same for the comparison. Batch size safely at 2 since
# 500M easily fits.
sft_cfg = SFTConfig(
    output_dir=str(OUTPUT_500M_V2),
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    logging_steps=10,
    save_strategy="steps",
    save_steps=50,
    save_total_limit=3,
    bf16=True,        # 500M can use bf16 like 1.5B
    optim="paged_adamw_8bit",
    report_to="none",
    max_length=768,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_ds,
    processing_class=tok,
    args=sft_cfg,
)

print("starting training...")
trainer.train()
print("done. saving...")
trainer.save_model(str(OUTPUT_500M_V2 / "checkpoint-final"))
print(f"saved to {OUTPUT_500M_V2}")

del model, base, trainer, tok
free_gpu()

VRAM: 0.00 GB
training on 1598 examples
dataset size: 1598


W0903 14:41:54.807000 11608 Lib\site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels


model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

C:\Users\shlok\projects\ddp-llm\.venv\Lib\site-packages\huggingface_hub\file_download.py:137: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\shlok\.cache\huggingface\hub\models--Qwen--Qwen2.5-0.5B-Instruct. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

C:\Users\shlok\projects\ddp-llm\.venv\Lib\site-packages\bitsandbytes\backends\cuda\ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


... (5 outputs left out)

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


starting training...


Step,Training Loss
10,2.857570
20,0.945808
30,0.074781
40,0.059558
50,0.054188
60,0.052967
70,0.047488
80,0.046820
90,0.046420
100,0.043797


done. saving...
saved to C:\Users\shlok\projects\ddp-llm\parser\checkpoints\qwen0p5b-qlora-v2
VRAM: 0.56 GB


In [5]:
free_gpu()

ADAPTER_500M_V2 = CHECKPOINTS / "qwen0p5b-qlora-v2_lite_reference" / "checkpoint-final"
assert ADAPTER_500M_V2.exists(), f"missing: {ADAPTER_500M_V2}"

test_data = load_jsonl(DATA / "test.jsonl")
print(f"loaded {len(test_data)} test examples")

# Same 70-case audit set used on 3B v2, so numbers are directly comparable.
ADVERSARIAL_FULL = [
    # -- group 1: quality/reaction words --
    ("works great",    "*"), ("looks nice",     "*"), ("looks good",     "*"),
    ("looks fine",     "*"), ("looks great",    "*"), ("seems nice",     "*"),
    ("cool",           "*"), ("nice one",       "*"), ("perfect",        "*"),
    ("great",          "*"), ("awesome",        "*"), ("amazing",        "*"),
    ("brilliant",      "*"), ("beautiful",      "*"), ("love it",        "*"),
    ("go with it",     "*"), ("that works",     "*"), ("that'll do",     "*"),
    ("solid",          "*"), ("neat",           "*"),

    # -- group 2: bare confirmations --
    ("yes",            "*"), ("yeah",           "*"), ("yep",            "*"),
    ("yup",            "*"), ("definitely",     "*"), ("absolutely",     "*"),
    ("absolutely yes", "*"), ("for sure",       "*"), ("sure",           "*"),
    ("of course",      "*"), ("100%",           "*"), ("totally",        "*"),
    ("exactly",        "*"), ("spot on",        "*"), ("right on",       "*"),
    ("bingo",          "*"), ("no doubt",       "*"), ("agreed",         "*"),

    # -- group 3: uncertainty / fillers --
    ("hmm",            "*"), ("okay",           "*"), ("meh",            "*"),
    ("not sure",       "*"), ("dunno really",   "*"), ("sure whatever",  "*"),
    ("interesting",    "*"), ("i guess",        "*"), ("alright",        "*"),

    # -- group 4: negative fillers --
    ("eh",             "*"), ("not great",      "*"), ("boring",         "*"),
    ("bland",          "*"), ("whatever",       "*"), ("nothing special","*"),

    # -- group 5: sanity (with referent) --
    ("A",              ["A"]), ("B",              ["B"]),
    ("yes A",          ["A"]), ("definitely B",   ["B"]),
    ("A definitely",   ["A"]), ("yeah B",         ["B"]),
    ("the left one",   ["A"]), ("the right one",  ["B"]),
    ("first please",   ["A"]), ("second please",  ["B"]),
    ("both are fine",  ["A", "B"]), ("either",     ["A", "B"]),
    ("A and B",        ["A", "B"]),
    ("neither",        []), ("none of them",   []),
    ("nope",           []), ("both wrong",     []),
]

print(f"\n{'='*60}\nloading 500M-v2...\n{'='*60}", flush=True)
model, tok = build_model(MODEL_500M, ADAPTER_500M_V2)
print(f"VRAM: {torch.cuda.memory_allocated() / 1e9:.2f} GB\n", flush=True)

# --- full test set ---
print(f"=== 500M-v2: standard test set ===")
correct = 0
per_cat = {}
failures = []
for i, ex in enumerate(test_data):
    if i % 40 == 0:
        print(f"  progress: {i}/{len(test_data)}", flush=True)
    parsed, raw = parse(model, tok, ex["utterance"], ex["options"])
    ok = labels_equal(parsed, ex["label"])
    cat = ex.get("category", "unknown")
    per_cat.setdefault(cat, [0, 0])
    per_cat[cat][1] += 1
    if ok:
        correct += 1
        per_cat[cat][0] += 1
    else:
        failures.append((ex, parsed, raw))
print(f"\noverall: {correct}/{len(test_data)} = {100*correct/len(test_data):.1f}%")
for cat in sorted(per_cat):
    c, t = per_cat[cat]
    print(f"  {cat:20s} {c}/{t} = {100*c/t:.1f}%")
if failures:
    print(f"failures ({len(failures)}):")
    for ex, parsed, raw in failures[:15]:
        print(f"  utt={ex['utterance']!r}  gold={ex['label']!r}  got={parsed!r}")

# --- adversarial audit ---
print(f"\n\n=== 500M-v2: full adversarial audit ({len(ADVERSARIAL_FULL)} cases) ===\n")
groups = {
    "1. quality/reaction":   ADVERSARIAL_FULL[0:20],
    "2. bare confirmations": ADVERSARIAL_FULL[20:38],
    "3. uncertainty/fillers": ADVERSARIAL_FULL[38:47],
    "4. negative fillers":   ADVERSARIAL_FULL[47:53],
    "5. sanity (referent)":  ADVERSARIAL_FULL[53:],
}
group_scores = {}
all_failures = []
for group_name, cases in groups.items():
    print(f"--- {group_name} ---")
    n_ok = 0
    for utt, gold in cases:
        parsed, raw = parse(model, tok, utt, ("A", "B"))
        ok = labels_equal(parsed, gold)
        if ok:
            n_ok += 1
        else:
            all_failures.append((group_name, utt, gold, parsed))
        mark = "OK  " if ok else "FAIL"
        print(f"  [{mark}] {utt!r:20s} gold={gold!r:14s} got={parsed!r}")
    group_scores[group_name] = (n_ok, len(cases))
    print(f"  -> {n_ok}/{len(cases)}\n")

print(f"\n=== 500M-v2 SUMMARY vs 3B-v2 (recall) ===")
# 3B-v2 numbers from earlier audit for direct comparison
recall_3b = {
    "1. quality/reaction":   (20, 20),
    "2. bare confirmations": (5, 18),
    "3. uncertainty/fillers": (9, 9),
    "4. negative fillers":   (5, 6),
    "5. sanity (referent)":  (17, 17),
}
print(f"{'group':<28s} {'500M v2':<12s} {'3B v2':<12s} {'delta':<10s}")
for g in group_scores:
    c500, t500 = group_scores[g]
    c3b, t3b = recall_3b[g]
    delta = (100*c500/t500) - (100*c3b/t3b)
    print(f"{g:<28s} {c500}/{t500} ({100*c500/t500:5.1f}%)  {c3b}/{t3b} ({100*c3b/t3b:5.1f}%)  {delta:+.1f}pp")

del model, tok
free_gpu()

VRAM: 0.56 GB
loaded 210 test examples

loading 500M-v2...


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

C:\Users\shlok\projects\ddp-llm\.venv\Lib\site-packages\bitsandbytes\backends\cuda\ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


VRAM: 1.06 GB

=== 500M-v2: standard test set ===
  progress: 0/210


C:\Users\shlok\projects\ddp-llm\.venv\Lib\site-packages\bitsandbytes\backends\cuda\ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


  progress: 40/210
  progress: 80/210
  progress: 120/210
  progress: 160/210
  progress: 200/210

overall: 199/210 = 94.8%
  comparative          26/26 = 100.0%
  mixed_sentiment      26/28 = 92.9%
  multi_positive       13/13 = 100.0%
  negation             54/61 = 88.5%
  off_topic            22/22 = 100.0%
  reject_all           29/29 = 100.0%
  single_positive      9/9 = 100.0%
  uncertainty          20/22 = 90.9%
failures (11):
  utt='everyone except B is a miss'  gold=['B']  got=['A', 'C', 'D']
  utt='everything except A and C is a candidate'  gold=['B', 'D']  got=['A', 'C']
  utt='A far, B close, C far, D close'  gold=['B', 'D']  got=['A', 'C']
  utt='torn on all four'  gold='*'  got=[]
  utt='the first, third, and fourth are all wrong'  gold=['B']  got=['B', 'C']
  utt='beats me'  gold='*'  got=['A']
  utt='not the third or the fourth'  gold=['A', 'B']  got=['A']
  utt='no on D, keep the rest'  gold=['A', 'B', 'C']  got=['A', 'B']
  utt='not the first or the third'  gold=['B',

In [6]:
free_gpu()

ADAPTER_1P5B_V2 = CHECKPOINTS / "qwen1p5b-qlora-v2_smaller" / "checkpoint-final"
assert ADAPTER_1P5B_V2.exists(), f"missing: {ADAPTER_1P5B_V2}"

test_data = load_jsonl(DATA / "test.jsonl")

print(f"\n{'='*60}\nloading 1.5B-v2...\n{'='*60}", flush=True)
model, tok = build_model("Qwen/Qwen2.5-1.5B-Instruct", ADAPTER_1P5B_V2)
print(f"VRAM: {torch.cuda.memory_allocated() / 1e9:.2f} GB\n", flush=True)

# --- test set ---
print(f"=== 1.5B-v2: standard test set ===")
correct = 0
per_cat = {}
failures = []
for i, ex in enumerate(test_data):
    if i % 40 == 0:
        print(f"  progress: {i}/{len(test_data)}", flush=True)
    parsed, raw = parse(model, tok, ex["utterance"], ex["options"])
    ok = labels_equal(parsed, ex["label"])
    cat = ex.get("category", "unknown")
    per_cat.setdefault(cat, [0, 0])
    per_cat[cat][1] += 1
    if ok:
        correct += 1
        per_cat[cat][0] += 1
    else:
        failures.append((ex, parsed, raw))
print(f"\noverall: {correct}/{len(test_data)} = {100*correct/len(test_data):.1f}%")
for cat in sorted(per_cat):
    c, t = per_cat[cat]
    print(f"  {cat:20s} {c}/{t} = {100*c/t:.1f}%")
if failures:
    print(f"failures ({len(failures)}):")
    for ex, parsed, raw in failures[:15]:
        print(f"  utt={ex['utterance']!r}  gold={ex['label']!r}  got={parsed!r}")

# --- adversarial ---
print(f"\n\n=== 1.5B-v2: full adversarial audit ({len(ADVERSARIAL_FULL)} cases) ===\n")
groups = {
    "1. quality/reaction":   ADVERSARIAL_FULL[0:20],
    "2. bare confirmations": ADVERSARIAL_FULL[20:38],
    "3. uncertainty/fillers": ADVERSARIAL_FULL[38:47],
    "4. negative fillers":   ADVERSARIAL_FULL[47:53],
    "5. sanity (referent)":  ADVERSARIAL_FULL[53:],
}
group_scores_1p5b = {}
for group_name, cases in groups.items():
    print(f"--- {group_name} ---")
    n_ok = 0
    for utt, gold in cases:
        parsed, raw = parse(model, tok, utt, ("A", "B"))
        ok = labels_equal(parsed, gold)
        if ok:
            n_ok += 1
        mark = "OK  " if ok else "FAIL"
        print(f"  [{mark}] {utt!r:20s} gold={gold!r:14s} got={parsed!r}")
    group_scores_1p5b[group_name] = (n_ok, len(cases))
    print(f"  -> {n_ok}/{len(cases)}\n")


# --- three-way summary ---
print(f"\n{'='*70}")
print(f"THREE-WAY COMPARISON: 500M v2 / 1.5B v2 / 3B v2")
print(f"{'='*70}\n")

# from earlier audits
scores_500m = {
    "1. quality/reaction":   (18, 20),
    "2. bare confirmations": (9, 18),
    "3. uncertainty/fillers": (8, 9),
    "4. negative fillers":   (6, 6),
    "5. sanity (referent)":  (15, 17),
}
scores_3b = {
    "1. quality/reaction":   (20, 20),
    "2. bare confirmations": (5, 18),
    "3. uncertainty/fillers": (9, 9),
    "4. negative fillers":   (5, 6),
    "5. sanity (referent)":  (17, 17),
}

print(f"{'group':<28s} {'500M':<14s} {'1.5B':<14s} {'3B':<14s}")
print("-" * 70)
for g in group_scores_1p5b:
    c500, t500 = scores_500m[g]
    c15, t15 = group_scores_1p5b[g]
    c3b, t3b = scores_3b[g]
    print(f"{g:<28s} {c500}/{t500} ({100*c500/t500:5.1f}%)  {c15}/{t15} ({100*c15/t15:5.1f}%)  {c3b}/{t3b} ({100*c3b/t3b:5.1f}%)")

total_500m = sum(c for c, _ in scores_500m.values())
total_1p5b = sum(c for c, _ in group_scores_1p5b.values())
total_3b = sum(c for c, _ in scores_3b.values())
total_cases = sum(t for _, t in scores_500m.values())
print("-" * 70)
print(f"{'TOTAL adversarial':<28s} {total_500m}/{total_cases} ({100*total_500m/total_cases:5.1f}%)  {total_1p5b}/{total_cases} ({100*total_1p5b/total_cases:5.1f}%)  {total_3b}/{total_cases} ({100*total_3b/total_cases:5.1f}%)")

del model, tok
free_gpu()

VRAM: 0.56 GB

loading 1.5B-v2...


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

VRAM: 1.79 GB

=== 1.5B-v2: standard test set ===
  progress: 0/210
  progress: 40/210
  progress: 80/210
  progress: 120/210
  progress: 160/210
  progress: 200/210

overall: 205/210 = 97.6%
  comparative          25/26 = 96.2%
  mixed_sentiment      27/28 = 96.4%
  multi_positive       12/13 = 92.3%
  negation             61/61 = 100.0%
  off_topic            22/22 = 100.0%
  reject_all           28/29 = 96.6%
  single_positive      8/9 = 88.9%
  uncertainty          22/22 = 100.0%
failures (5):
  utt='B leaves D in the dust'  gold=['B']  got=['A', 'C']
  utt='A perfect, B terrible, C okay, D terrible'  gold=['A', 'C']  got=['A']
  utt='is it A?'  gold=['A']  got='*'
  utt='none is the target'  gold=[]  got='*'
  utt='A is okay but B is closer'  gold=['A', 'B']  got=['B']


=== 1.5B-v2: full adversarial audit (70 cases) ===

--- 1. quality/reaction ---
  [FAIL] 'works great'        gold='*'            got=['Works great']
  [OK  ] 'looks nice'         gold='*'            got='*'
  [FA

In [7]:
import os
from datetime import datetime

def when(p):
    if p.exists():
        return datetime.fromtimestamp(os.path.getmtime(p)).strftime('%Y-%m-%d %H:%M:%S')
    return "does not exist"

print("adapter timestamps:")
print(f"  1.5B v2 checkpoint-final: {when(CHECKPOINTS / 'qwen1p5b-qlora-v2_smaller' / 'checkpoint-final')}")
print(f"  3B v2 checkpoint-final:   {when(CHECKPOINTS / 'qwen3b-qlora-v2' / 'checkpoint-final')}")
print(f"  500M v2 checkpoint-final: {when(CHECKPOINTS / 'qwen0p5b-qlora-v2_lite_reference' / 'checkpoint-final')}")

# also check the train.jsonl backups
print("\ntrain.jsonl backup timestamps + line counts:")
for p in sorted(DATA.glob("train_v*.jsonl")):
    n = sum(1 for _ in p.open(encoding="utf-8"))
    print(f"  {p.name}: {n} lines, mtime {when(p)}")


adapter timestamps:
  1.5B v2 checkpoint-final: 2026-09-02 17:23:37
  3B v2 checkpoint-final:   2026-09-02 19:00:35
  500M v2 checkpoint-final: 2026-09-03 15:19:31

train.jsonl backup timestamps + line counts:
  train.jsonl.bak-20260902-162518: 1278 lines, mtime 2026-09-02 16:25:18
  train.jsonl.bak-20260903-002204: 1451 lines, mtime 2026-09-03 00:22:04
  current train.jsonl: 1598 lines


In [8]:
free_gpu()

MODEL_1P5B = "Qwen/Qwen2.5-1.5B-Instruct"
# Fresh folder for the new run so we can compare v2 vs v3
OUTPUT_1P5B_V3 = CHECKPOINTS / "qwen1p5b-qlora-v3_confirmation_win"

train_data_full = load_jsonl(DATA / "train_v3.jsonl")
print(f"training on {len(train_data_full)} examples")

def to_chat(ex):
    options_str = "[" + ", ".join(f'"{o}"' for o in ex["options"]) + "]"
    user_msg = f"Options: {options_str}\nUser: {ex['utterance']}"
    assistant_msg = json.dumps(ex["label"])
    return {"messages": [
        {"role": "system", "content": SYSTEM_PROMPT_V3},
        {"role": "user", "content": user_msg},
        {"role": "assistant", "content": assistant_msg},
    ]}

train_ds = Dataset.from_list([to_chat(ex) for ex in train_data_full])

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)
tok = AutoTokenizer.from_pretrained(MODEL_1P5B)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

base = AutoModelForCausalLM.from_pretrained(
    MODEL_1P5B, quantization_config=bnb, device_map="cuda",
)
base = prepare_model_for_kbit_training(base)

lora_cfg = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
)
model = get_peft_model(base, lora_cfg)
model.print_trainable_parameters()

# 3 epochs (same as 500M v2 for consistency). Same safer batch settings.
sft_cfg = SFTConfig(
    output_dir=str(OUTPUT_1P5B_V3),
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    logging_steps=10,
    save_strategy="steps",
    save_steps=50,
    save_total_limit=3,
    bf16=True,
    optim="paged_adamw_8bit",
    report_to="none",
    max_length=768,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_ds,
    processing_class=tok,
    args=sft_cfg,
)

print("starting training...")
trainer.train()
print("done. saving...")
trainer.save_model(str(OUTPUT_1P5B_V3 / "checkpoint-final"))
print(f"saved to {OUTPUT_1P5B_V3}")

del model, base, trainer, tok
free_gpu()

VRAM: 0.56 GB
training on 1598 examples


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


trainable params: 18,464,768 || all params: 1,562,179,072 || trainable%: 1.1820


Tokenizing train dataset:   0%|          | 0/1598 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/1598 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


starting training...


Step,Training Loss
10,2.550211
20,0.867455
30,0.063789
40,0.055097
50,0.050855
60,0.049631
70,0.044942
80,0.044312
90,0.044564
100,0.041662


done. saving...
saved to C:\Users\shlok\projects\ddp-llm\parser\checkpoints\qwen1.5b-qlora-v3
VRAM: 1.50 GB


In [9]:
free_gpu()

ADAPTER_1P5B_V3 = CHECKPOINTS / "qwen1p5b-qlora-v3_confirmation_win" / "checkpoint-final"
assert ADAPTER_1P5B_V3.exists(), f"missing: {ADAPTER_1P5B_V3}"

test_data = load_jsonl(DATA / "test.jsonl")

print(f"\n{'='*60}\nloading 1.5B-v3...\n{'='*60}", flush=True)
model, tok = build_model("Qwen/Qwen2.5-1.5B-Instruct", ADAPTER_1P5B_V3)
print(f"VRAM: {torch.cuda.memory_allocated() / 1e9:.2f} GB\n", flush=True)

# --- test set ---
print(f"=== 1.5B-v3: standard test set ===")
correct = 0
per_cat = {}
failures = []
for i, ex in enumerate(test_data):
    if i % 40 == 0:
        print(f"  progress: {i}/{len(test_data)}", flush=True)
    parsed, raw = parse(model, tok, ex["utterance"], ex["options"])
    ok = labels_equal(parsed, ex["label"])
    cat = ex.get("category", "unknown")
    per_cat.setdefault(cat, [0, 0])
    per_cat[cat][1] += 1
    if ok:
        correct += 1
        per_cat[cat][0] += 1
    else:
        failures.append((ex, parsed, raw))
print(f"\noverall: {correct}/{len(test_data)} = {100*correct/len(test_data):.1f}%")
for cat in sorted(per_cat):
    c, t = per_cat[cat]
    print(f"  {cat:20s} {c}/{t} = {100*c/t:.1f}%")
if failures:
    print(f"failures ({len(failures)}):")
    for ex, parsed, raw in failures[:15]:
        print(f"  utt={ex['utterance']!r}  gold={ex['label']!r}  got={parsed!r}")

# --- adversarial ---
print(f"\n\n=== 1.5B-v3: full adversarial audit ({len(ADVERSARIAL_FULL)} cases) ===\n")
groups = {
    "1. quality/reaction":   ADVERSARIAL_FULL[0:20],
    "2. bare confirmations": ADVERSARIAL_FULL[20:38],
    "3. uncertainty/fillers": ADVERSARIAL_FULL[38:47],
    "4. negative fillers":   ADVERSARIAL_FULL[47:53],
    "5. sanity (referent)":  ADVERSARIAL_FULL[53:],
}
group_scores_1p5b_v3 = {}
for group_name, cases in groups.items():
    print(f"--- {group_name} ---")
    n_ok = 0
    for utt, gold in cases:
        parsed, raw = parse(model, tok, utt, ("A", "B"))
        ok = labels_equal(parsed, gold)
        if ok:
            n_ok += 1
        mark = "OK  " if ok else "FAIL"
        print(f"  [{mark}] {utt!r:20s} gold={gold!r:14s} got={parsed!r}")
    group_scores_1p5b_v3[group_name] = (n_ok, len(cases))
    print(f"  -> {n_ok}/{len(cases)}\n")


# --- four-way summary: 500M v2 / 1.5B v2 / 1.5B v3 / 3B v2 ---
print(f"\n{'='*82}")
print(f"FOUR-WAY COMPARISON")
print(f"{'='*82}\n")

scores_500m = {
    "1. quality/reaction":   (18, 20),
    "2. bare confirmations": (9, 18),
    "3. uncertainty/fillers": (8, 9),
    "4. negative fillers":   (6, 6),
    "5. sanity (referent)":  (15, 17),
}
scores_1p5b_v2 = {
    "1. quality/reaction":   (12, 20),
    "2. bare confirmations": (1, 18),
    "3. uncertainty/fillers": (9, 9),
    "4. negative fillers":   (6, 6),
    "5. sanity (referent)":  (16, 17),
}
scores_3b = {
    "1. quality/reaction":   (20, 20),
    "2. bare confirmations": (5, 18),
    "3. uncertainty/fillers": (9, 9),
    "4. negative fillers":   (5, 6),
    "5. sanity (referent)":  (17, 17),
}

print(f"{'group':<28s} {'500M v2':<14s} {'1.5B v2':<14s} {'1.5B v3':<14s} {'3B v2':<14s}")
print("-" * 82)
for g in group_scores_1p5b_v3:
    c500, t500 = scores_500m[g]
    c15v2, t15v2 = scores_1p5b_v2[g]
    c15v3, t15v3 = group_scores_1p5b_v3[g]
    c3b, t3b = scores_3b[g]
    print(f"{g:<28s} {c500}/{t500} ({100*c500/t500:5.1f}%)  {c15v2}/{t15v2} ({100*c15v2/t15v2:5.1f}%)  {c15v3}/{t15v3} ({100*c15v3/t15v3:5.1f}%)  {c3b}/{t3b} ({100*c3b/t3b:5.1f}%)")

total_500 = sum(c for c, _ in scores_500m.values())
total_15v2 = sum(c for c, _ in scores_1p5b_v2.values())
total_15v3 = sum(c for c, _ in group_scores_1p5b_v3.values())
total_3b = sum(c for c, _ in scores_3b.values())
total_cases = sum(t for _, t in scores_500m.values())
print("-" * 82)
print(f"{'TOTAL adversarial':<28s} {total_500}/{total_cases} ({100*total_500/total_cases:5.1f}%)  {total_15v2}/{total_cases} ({100*total_15v2/total_cases:5.1f}%)  {total_15v3}/{total_cases} ({100*total_15v3/total_cases:5.1f}%)  {total_3b}/{total_cases} ({100*total_3b/total_cases:5.1f}%)")

del model, tok
free_gpu()


VRAM: 1.50 GB

loading 1.5B-v3...


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

C:\Users\shlok\projects\ddp-llm\.venv\Lib\site-packages\bitsandbytes\backends\cuda\ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


VRAM: 2.73 GB

=== 1.5B-v3: standard test set ===
  progress: 0/210


C:\Users\shlok\projects\ddp-llm\.venv\Lib\site-packages\bitsandbytes\backends\cuda\ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


  progress: 40/210
  progress: 80/210
  progress: 120/210
  progress: 160/210
  progress: 200/210

overall: 207/210 = 98.6%
  comparative          25/26 = 96.2%
  mixed_sentiment      27/28 = 96.4%
  multi_positive       12/13 = 92.3%
  negation             61/61 = 100.0%
  off_topic            22/22 = 100.0%
  reject_all           29/29 = 100.0%
  single_positive      9/9 = 100.0%
  uncertainty          22/22 = 100.0%
failures (3):
  utt='C and D are both in the running, C edges it out'  gold=['C', 'D']  got=['C']
  utt='A perfect, B terrible, C okay, D terrible'  gold=['A', 'C']  got=['A']
  utt='A is okay but B is closer'  gold=['A', 'B']  got=['B']


=== 1.5B-v3: full adversarial audit (70 cases) ===

--- 1. quality/reaction ---
  [FAIL] 'works great'        gold='*'            got=['A', 'B']
  [FAIL] 'looks nice'         gold='*'            got=['A', 'B']
  [FAIL] 'looks good'         gold='*'            got=['A', 'B']
  [FAIL] 'looks fine'         gold='*'            got=['A', 'B

In [10]:
free_gpu()

MODEL_3B = "Qwen/Qwen2.5-3B-Instruct"
# NOTE: previous 3B v3 folder already exists as qwen3b-qlora-v3_bad_do_not_use
# (renamed after the 4-epoch + v4-prompt attempt failed).
# Using v3_clean to distinguish this new run.
OUTPUT_3B_V3_CLEAN = CHECKPOINTS / "qwen3b-qlora-v3_no_improvement"

train_data_full = load_jsonl(DATA / "train_v3.jsonl")
print(f"training on {len(train_data_full)} examples")

def to_chat(ex):
    options_str = "[" + ", ".join(f'"{o}"' for o in ex["options"]) + "]"
    user_msg = f"Options: {options_str}\nUser: {ex['utterance']}"
    assistant_msg = json.dumps(ex["label"])
    return {"messages": [
        {"role": "system", "content": SYSTEM_PROMPT_V3},
        {"role": "user", "content": user_msg},
        {"role": "assistant", "content": assistant_msg},
    ]}

train_ds = Dataset.from_list([to_chat(ex) for ex in train_data_full])

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)
tok = AutoTokenizer.from_pretrained(MODEL_3B)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

base = AutoModelForCausalLM.from_pretrained(
    MODEL_3B, quantization_config=bnb, device_map="cuda",
)
base = prepare_model_for_kbit_training(base)

lora_cfg = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
)
model = get_peft_model(base, lora_cfg)
model.print_trainable_parameters()

sft_cfg = SFTConfig(
    output_dir=str(OUTPUT_3B_V3_CLEAN),
    num_train_epochs=3,                        # KEY: 3 not 4
    per_device_train_batch_size=1,             # 3B on 8GB VRAM
    gradient_accumulation_steps=16,            # eff batch 16
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    logging_steps=10,
    save_strategy="steps",
    save_steps=50,
    save_total_limit=3,
    bf16=False,                                # Blackwell fix for 3B
    optim="paged_adamw_8bit",
    report_to="none",
    max_length=768,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_ds,
    processing_class=tok,
    args=sft_cfg,
)

print("starting training...")
trainer.train()
print("done. saving...")
trainer.save_model(str(OUTPUT_3B_V3_CLEAN / "checkpoint-final"))
print(f"saved to {OUTPUT_3B_V3_CLEAN}")

del model, base, trainer, tok
free_gpu()

VRAM: 1.50 GB
training on 1598 examples


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


trainable params: 29,933,568 || all params: 3,115,872,256 || trainable%: 0.9607


Tokenizing train dataset:   0%|          | 0/1598 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/1598 [00:00<?, ? examples/s]

... (3 outputs left out)

C:\Users\shlok\projects\ddp-llm\.venv\Lib\site-packages\peft\utils\other.py:1419: UserWarning: Unable to fetch remote file due to the following error timed out - silently ignoring the lookup for the file config.json in Qwen/Qwen2.5-3B-Instruct.
  warnings.warn(
C:\Users\shlok\projects\ddp-llm\.venv\Lib\site-packages\peft\utils\save_and_load.py:372: UserWarning: Could not find a config file in Qwen/Qwen2.5-3B-Instruct - will assume that the vocabulary was not modified.
  warnings.warn(
C:\Users\shlok\projects\ddp-llm\.venv\Lib\site-packages\peft\utils\other.py:1419: UserWarning: Unable to fetch remote file due to the following error timed out - silently ignoring the lookup for the file config.json in Qwen/Qwen2.5-3B-Instruct.
  warnings.warn(
C:\Users\shlok\projects\ddp-llm\.venv\Lib\site-packages\peft\utils\save_and_load.py:372: UserWarning: Could not find a config file in Qwen/Qwen2.5-3B-Instruct - will assume that the vocabulary was not modified.
  warnings.warn(
C:\Users\shlok\proje

done. saving...


C:\Users\shlok\projects\ddp-llm\.venv\Lib\site-packages\peft\utils\other.py:1419: UserWarning: Unable to fetch remote file due to the following error timed out - silently ignoring the lookup for the file config.json in Qwen/Qwen2.5-3B-Instruct.
  warnings.warn(
C:\Users\shlok\projects\ddp-llm\.venv\Lib\site-packages\peft\utils\save_and_load.py:372: UserWarning: Could not find a config file in Qwen/Qwen2.5-3B-Instruct - will assume that the vocabulary was not modified.
  warnings.warn(


saved to C:\Users\shlok\projects\ddp-llm\parser\checkpoints\qwen3b-qlora-v3_clean
VRAM: 2.74 GB


In [11]:
free_gpu()

ADAPTER_3B_V3_CLEAN = CHECKPOINTS / "qwen3b-qlora-v3_no_improvement" / "checkpoint-final"
assert ADAPTER_3B_V3_CLEAN.exists(), f"missing: {ADAPTER_3B_V3_CLEAN}"

test_data = load_jsonl(DATA / "test.jsonl")

print(f"\n{'='*60}\nloading 3B-v3_clean...\n{'='*60}", flush=True)
model, tok = build_model("Qwen/Qwen2.5-3B-Instruct", ADAPTER_3B_V3_CLEAN)
print(f"VRAM: {torch.cuda.memory_allocated() / 1e9:.2f} GB\n", flush=True)

# --- test set ---
print(f"=== 3B-v3_clean: standard test set ===")
correct = 0
per_cat = {}
failures = []
for i, ex in enumerate(test_data):
    if i % 40 == 0:
        print(f"  progress: {i}/{len(test_data)}", flush=True)
    parsed, raw = parse(model, tok, ex["utterance"], ex["options"])
    ok = labels_equal(parsed, ex["label"])
    cat = ex.get("category", "unknown")
    per_cat.setdefault(cat, [0, 0])
    per_cat[cat][1] += 1
    if ok:
        correct += 1
        per_cat[cat][0] += 1
    else:
        failures.append((ex, parsed, raw))
print(f"\noverall: {correct}/{len(test_data)} = {100*correct/len(test_data):.1f}%")
for cat in sorted(per_cat):
    c, t = per_cat[cat]
    print(f"  {cat:20s} {c}/{t} = {100*c/t:.1f}%")
if failures:
    print(f"failures ({len(failures)}):")
    for ex, parsed, raw in failures[:15]:
        print(f"  utt={ex['utterance']!r}  gold={ex['label']!r}  got={parsed!r}")

# --- adversarial ---
print(f"\n\n=== 3B-v3_clean: full adversarial audit ({len(ADVERSARIAL_FULL)} cases) ===\n")
groups = {
    "1. quality/reaction":   ADVERSARIAL_FULL[0:20],
    "2. bare confirmations": ADVERSARIAL_FULL[20:38],
    "3. uncertainty/fillers": ADVERSARIAL_FULL[38:47],
    "4. negative fillers":   ADVERSARIAL_FULL[47:53],
    "5. sanity (referent)":  ADVERSARIAL_FULL[53:],
}
group_scores_3b_v3 = {}
for group_name, cases in groups.items():
    print(f"--- {group_name} ---")
    n_ok = 0
    for utt, gold in cases:
        parsed, raw = parse(model, tok, utt, ("A", "B"))
        ok = labels_equal(parsed, gold)
        if ok:
            n_ok += 1
        mark = "OK  " if ok else "FAIL"
        print(f"  [{mark}] {utt!r:20s} gold={gold!r:14s} got={parsed!r}")
    group_scores_3b_v3[group_name] = (n_ok, len(cases))
    print(f"  -> {n_ok}/{len(cases)}\n")


# --- five-way summary ---
print(f"\n{'='*98}")
print(f"FIVE-WAY COMPARISON")
print(f"{'='*98}\n")

scores_500m = {
    "1. quality/reaction":   (18, 20),
    "2. bare confirmations": (9, 18),
    "3. uncertainty/fillers": (8, 9),
    "4. negative fillers":   (6, 6),
    "5. sanity (referent)":  (15, 17),
}
scores_1p5b_v2 = {
    "1. quality/reaction":   (12, 20),
    "2. bare confirmations": (1, 18),
    "3. uncertainty/fillers": (9, 9),
    "4. negative fillers":   (6, 6),
    "5. sanity (referent)":  (16, 17),
}
scores_1p5b_v3 = {
    "1. quality/reaction":   (12, 20),
    "2. bare confirmations": (12, 18),
    "3. uncertainty/fillers": (9, 9),
    "4. negative fillers":   (5, 6),
    "5. sanity (referent)":  (17, 17),
}
scores_3b_v2 = {
    "1. quality/reaction":   (20, 20),
    "2. bare confirmations": (5, 18),
    "3. uncertainty/fillers": (9, 9),
    "4. negative fillers":   (5, 6),
    "5. sanity (referent)":  (17, 17),
}

print(f"{'group':<28s} {'500M v2':<14s} {'1.5B v2':<14s} {'1.5B v3':<14s} {'3B v2':<14s} {'3B v3':<14s}")
print("-" * 98)
for g in group_scores_3b_v3:
    c500, t500 = scores_500m[g]
    c15v2, t15v2 = scores_1p5b_v2[g]
    c15v3, t15v3 = scores_1p5b_v3[g]
    c3bv2, t3bv2 = scores_3b_v2[g]
    c3bv3, t3bv3 = group_scores_3b_v3[g]
    print(f"{g:<28s} {c500}/{t500} ({100*c500/t500:5.1f}%)  {c15v2}/{t15v2} ({100*c15v2/t15v2:5.1f}%)  {c15v3}/{t15v3} ({100*c15v3/t15v3:5.1f}%)  {c3bv2}/{t3bv2} ({100*c3bv2/t3bv2:5.1f}%)  {c3bv3}/{t3bv3} ({100*c3bv3/t3bv3:5.1f}%)")

total_500 = sum(c for c, _ in scores_500m.values())
total_15v2 = sum(c for c, _ in scores_1p5b_v2.values())
total_15v3 = sum(c for c, _ in scores_1p5b_v3.values())
total_3bv2 = sum(c for c, _ in scores_3b_v2.values())
total_3bv3 = sum(c for c, _ in group_scores_3b_v3.values())
total_cases = sum(t for _, t in scores_500m.values())
print("-" * 98)
print(f"{'TOTAL adversarial':<28s} {total_500}/{total_cases} ({100*total_500/total_cases:5.1f}%)  {total_15v2}/{total_cases} ({100*total_15v2/total_cases:5.1f}%)  {total_15v3}/{total_cases} ({100*total_15v3/total_cases:5.1f}%)  {total_3bv2}/{total_cases} ({100*total_3bv2/total_cases:5.1f}%)  {total_3bv3}/{total_cases} ({100*total_3bv3/total_cases:5.1f}%)")

del model, tok
free_gpu()

VRAM: 2.74 GB

loading 3B-v3_clean...


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

C:\Users\shlok\projects\ddp-llm\.venv\Lib\site-packages\bitsandbytes\backends\cuda\ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


VRAM: 4.93 GB

=== 3B-v3_clean: standard test set ===
  progress: 0/210


C:\Users\shlok\projects\ddp-llm\.venv\Lib\site-packages\bitsandbytes\backends\cuda\ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


  progress: 40/210
  progress: 80/210
  progress: 120/210
  progress: 160/210
  progress: 200/210

overall: 209/210 = 99.5%
  comparative          25/26 = 96.2%
  mixed_sentiment      28/28 = 100.0%
  multi_positive       13/13 = 100.0%
  negation             61/61 = 100.0%
  off_topic            22/22 = 100.0%
  reject_all           29/29 = 100.0%
  single_positive      9/9 = 100.0%
  uncertainty          22/22 = 100.0%
failures (1):
  utt='C and D are both in the running, C edges it out'  gold=['C', 'D']  got=['C']


=== 3B-v3_clean: full adversarial audit (70 cases) ===

--- 1. quality/reaction ---
  [OK  ] 'works great'        gold='*'            got='*'
  [OK  ] 'looks nice'         gold='*'            got='*'
  [FAIL] 'looks good'         gold='*'            got=['A']
  [FAIL] 'looks fine'         gold='*'            got=['A']
  [OK  ] 'looks great'        gold='*'            got='*'
  [OK  ] 'seems nice'         gold='*'            got='*'
  [OK  ] 'cool'               gold='*' 